In [1]:
"""
process_matches_glicko2.py

Two processing loops for 05_glicko2.ipynb, matching the real Setka.csv columns
(Date, Player1, Player2, Sets_P1, Sets_P2, HomeWinner) and the DataFrame-based
create_player_base() convention already used in utils.py.

  - predict BEFORE updating (no leakage)
  - dynamic results filename
  - fresh player_base built per run
  - same NaN-row cleaning as the Elo notebooks (dropna on Sets_P1/Sets_P2)

Option 1: per-match periods  -> each match is its own rating period, updated immediately.
Option 2: batched periods    -> matches grouped by Date (day/week/etc); ratings update once
                                 per period; players inactive in a period still get their
                                 RD inflated per the Glicko-2 spec.
"""

import pandas as pd
import sys
sys.path.append('..')
from src.glicko2_utils import (
    create_glicko_player_base,
    player_base_to_dict,
    dict_to_player_base,
    predict_probability,
    update_player_one_period,
)

In [2]:
def _load_and_clean(all_matches):
    """Same cleaning step as the Elo notebooks: drop rows with missing sets,
    parse Date, sort chronologically (Glicko-2 periods only make sense in order)."""
    df = all_matches.dropna(subset=["Sets_P1", "Sets_P2"]).copy()
    df["Date"] = pd.to_datetime(df["Date"])
    return df.sort_values("Date").reset_index(drop=True)

In [3]:
#---------------------------------------------------------------------------
# Option 1: per-match rating periods
# ---------------------------------------------------------------------------

def process_matches_glicko2_permatch(all_matches):
    """
    Each match = its own rating period for both players involved.
    Returns (results_df, final_player_base_df) -- final_player_base_df has the
    same shape create_player_base() would give you, plus rd/vol columns.
    """
    df = _load_and_clean(all_matches)
    player_base_df = create_glicko_player_base(df)
    player_base = player_base_to_dict(player_base_df)

    results = []

    for _, match in df.iterrows():
        p1, p2 = match["Player1"], match["Player2"]
        a, b = player_base[p1], player_base[p2]

        # Predict BEFORE updating either player
        win_prob_p1 = predict_probability(a["rating"], a["rd"], b["rating"], b["rd"])

        actual_outcome = 1 if match["HomeWinner"] == 1 else 0
        score_p1 = float(actual_outcome)
        score_p2 = 1.0 - score_p1

        new_r1, new_rd1, new_vol1 = update_player_one_period(
            a["rating"], a["rd"], a["vol"], [(b["rating"], b["rd"], score_p1)]
        )
        new_r2, new_rd2, new_vol2 = update_player_one_period(
            b["rating"], b["rd"], b["vol"], [(a["rating"], a["rd"], score_p2)]
        )

        player_base[p1] = {
            "rating": new_r1, "rd": new_rd1, "vol": new_vol1,
            "matches_played": a["matches_played"] + 1,
        }
        player_base[p2] = {
            "rating": new_r2, "rd": new_rd2, "vol": new_vol2,
            "matches_played": b["matches_played"] + 1,
        }

        results.append({
            "Date": match["Date"],
            "Player1": p1,
            "Player2": p2,
            "win_probability_p1": round(win_prob_p1, 4),
            "actual_outcome": actual_outcome,
        })

    return pd.DataFrame(results), dict_to_player_base(player_base)


In [4]:
# ---------------------------------------------------------------------------
# Option 2: batched rating periods
# ---------------------------------------------------------------------------

def process_matches_glicko2_batched(all_matches, period="D"):
    """
    Groups matches into rating periods using the Date column (pandas offset
    alias: 'D' daily, 'W' weekly, etc). Within a period:
      1. Predictions for every match use the rating/RD/vol as of the START of
         the period (pre-update snapshot) -- no leakage even for two matches
         by the same player within one period.
      2. All opponent results for a player in that period are collected, then
         ONE Glicko-2 update is applied per player at period end.
      3. Players with no match in a given period still get their RD inflated
         per the spec (rating/vol untouched) -- this is what makes batched
         Glicko-2 different from per-match: inactivity costs you certainty.

    Setka.csv's Date column includes time-of-day (matches every ~15 min in
    your sample), so daily buckets ('D') group many matches per period --
    weekly ('W') would group even more. Pick based on how "instant" you want
    rank movement to feel vs. how much data you have per bucket.
    """
    df = _load_and_clean(all_matches)
    df["_period"] = df["Date"].dt.to_period(period)

    player_base_df = create_glicko_player_base(df)
    player_base = player_base_to_dict(player_base_df)

    results = []

    for period_key, period_matches in df.groupby("_period", sort=True):
        # snapshot ratings as of period start -- used for ALL predictions this period
        snapshot = {p: dict(v) for p, v in player_base.items()}

        period_opponents = {p: [] for p in player_base}

        for _, match in period_matches.iterrows():
            p1, p2 = match["Player1"], match["Player2"]
            a, b = snapshot[p1], snapshot[p2]

            win_prob_p1 = predict_probability(a["rating"], a["rd"], b["rating"], b["rd"])

            actual_outcome = 1 if match["HomeWinner"] == 1 else 0
            score_p1 = float(actual_outcome)
            score_p2 = 1.0 - score_p1

            period_opponents[p1].append((b["rating"], b["rd"], score_p1))
            period_opponents[p2].append((a["rating"], a["rd"], score_p2))

            results.append({
                "Date": match["Date"],
                "period": str(period_key),
                "Player1": p1,
                "Player2": p2,
                "win_probability_p1": round(win_prob_p1, 4),
                "actual_outcome": actual_outcome,
            })

        # apply ONE update per player for this period (active or not)
        for p, state in player_base.items():
            opponents = period_opponents.get(p, [])
            new_r, new_rd, new_vol = update_player_one_period(
                state["rating"], state["rd"], state["vol"], opponents
            )
            player_base[p] = {
                "rating": new_r,
                "rd": new_rd,
                "vol": new_vol,
                "matches_played": state["matches_played"] + len(opponents),
            }

    return pd.DataFrame(results), dict_to_player_base(player_base)

In [ ]:

all_matches = pd.read_csv('../data/Setka.csv')

results_permatch, base_permatch = process_matches_glicko2_permatch(all_matches)
results_permatch.to_csv('../results/glicko2_permatch_results.csv', index=False)
base_permatch.to_csv('../data/glicko2_permatch_player_base.csv', index=False)


results_batched, base_batched = process_matches_glicko2_batched(all_matches, period='D')
results_batched.to_csv('../results/glicko2_batched_D_results.csv', index=False)
base_batched.to_csv('../data/glicko2_batched_D_player_base.csv', index=False)